In [1]:
import boto3
import json
import pandas as pd
from tqdm import tqdm

# === Load Economic Data ===
def load_economic_data(filepath):
    df = pd.read_csv(filepath)
    df["Date"] = pd.to_datetime(df["Date"])  # Ensure Date is in datetime format
    return df

# === Load Daily Fed Funds Target Rate Data ===
def load_fed_funds_data(filepath):
    df = pd.read_csv(filepath)
    df["Date"] = pd.to_datetime(df["Date"])  # Ensure Date is in datetime format
    return df


In [2]:
# 📋 List of Prompt Variants to Test with Claude
prompt_variants = [
    # Style 1 – Analyst
    """
    You are an experienced financial analyst. Use the following {n} months of U.S. macroeconomic indicators to forecast the Federal Funds Target Rate for {target_date}.

    Please respond in valid JSON with the following fields:
    {{
      "predicted_rate": "e.g. 5.25",
      "confidence": "e.g. 85%",
      "explanation": "Brief rationale, max 100 words"
    }}

    {data_table}
    """,

    # Style 2 – Central Bank Advisor
    """
    As an AI central bank advisor, review the following {n} months of U.S. economic data and predict the Fed Funds Target Rate the FOMC is likely to set on {target_date}.

    Reply in strict JSON:
    {{
      "predicted_rate": "...",
      "confidence": "...",
      "explanation": "..."
    }}

    Data provided below:
    {data_table}
    """,

    # Style 3 – Quantitative Model
    """
    Using the structured economic data below from the past {n} months, estimate the Fed Funds Target Rate for {target_date}. Focus on trends in inflation, labor market conditions, and money supply.

    Return only a JSON object:
    {{
      "predicted_rate": "numeric string",
      "confidence": "percent string",
      "explanation": "concise reasoning"
    }}

    Input data:
    {data_table}
    """,

    # Style 4 – Policy Simulation
    """
    Simulate a Federal Reserve policy discussion. Given the following economic indicators from the last {n} months, what decision is most likely regarding the Fed Funds Rate on {target_date}?

    Respond with this JSON structure only:
    {{
      "predicted_rate": "...",
      "confidence": "...",
      "explanation": "..."
    }}

    Indicators:
    {data_table}
    """,

    # Style 5 – Instructional Command
    """
    Predict the Federal Funds Target Rate decision for {target_date} based on this table of economic data from the last {n} months. Consider inflation, employment, and growth.

    Return a JSON object in the format:
    {{
      "predicted_rate": "5.25",
      "confidence": "90%",
      "explanation": "short reasoning"
    }}

    {data_table}
    """
]


def format_prompt(data, target_date, n):
    """
    Formats historical data into a structured prompt for Claude.

    Parameters:
        data (DataFrame): The historical economic indicators.
        target_date (Timestamp): The date of the Fed Funds Target Rate.
        n (int): Number of months to look back.

    Returns:
        str: Formatted prompt.
    """
    # Filter only data available before the target date
    filtered_data = data[data["Date"] < target_date].sort_values(by="Date", ascending=False).head(n)

    prompt = f"""
    Human: Based on the following {n} months of historical economic indicators, predict the Fed Funds Target Rate.

    Data:
    {filtered_data.to_string(index=False)}

    Provide the output in JSON format:
    {{
        "predicted_rate": ["predicted rate in percentage"],
        "confidence": ["percentage confidence"],
        "explanation": ["max 100 words"]
    }}

    Assistant:
    """
    return prompt.strip()


In [3]:
def format_prompt(data, target_date, n, template=None):
    """
    Formats the Claude prompt using the provided data.

    Parameters:
        data (DataFrame): The n months of economic data
        target_date (datetime): The prediction date
        n (int): Number of months used
        template (str): Optional custom prompt template with placeholders

    Returns:
        str: Full prompt string
    """
    data_table = data.to_markdown(index=False)

    if template is None:
        # Default fallback template
        template = f"""
You are an AI economist. Based on the following {n} months of U.S. macroeconomic data,
predict the Federal Funds Target Rate for {target_date.date()}.

Respond in this JSON format:
{{
  "predicted_rate": "...",
  "confidence": "...",
  "explanation": "..."
}}

{data_table}
"""
    else:
        template = template.format(n=n, target_date=str(target_date), data_table=data_table)

    return template


In [4]:
# 🔍 Function: Search best prompt from multiple prompt variants
def search_best_prompt_variants(prompt_variants, economic_df, fed_funds_df, n=6):
    """
    Tests multiple prompt templates and evaluates which gives the best predictive performance.
    
    Parameters:
        prompt_variants (list of str): Different prompt template strings
        economic_df (DataFrame): Economic data
        fed_funds_df (DataFrame): Actual Fed rate data
        n (int): Number of lookback months
    
    Returns:
        DataFrame: Ranked prompt templates and their average error
    """
    from tqdm import tqdm
    results = []

    # Limit to a few dates for testing — can increase later
    test_dates = fed_funds_df['Date'].sort_values().unique()[:36]

    for i, prompt_template in enumerate(prompt_variants):
        total_error = 0
        count = 0

        for date in test_dates:
            actual_row = fed_funds_df[fed_funds_df['Date'] == date]
            if actual_row.empty:
                continue
            actual_rate = actual_row.iloc[0]['Target_Rate']

            past_data = economic_df[economic_df['Date'] < date].sort_values("Date", ascending=False)
            filtered_data = past_data.head(n)
            if filtered_data.empty:
                continue

            prompt = format_prompt(filtered_data, date, n, template=prompt_template)
            response = query_claude_aws(prompt)

            try:
                predicted = float(response.get("predicted_rate", 0))
                error = abs(predicted - actual_rate)
                total_error += error
                count += 1
            except:
                continue

        avg_error = total_error / count if count else float('inf')
        results.append({
            "Prompt Index": i + 1,
            "Avg Error": avg_error,
            "Prompt": prompt_template
        })

    return pd.DataFrame(results).sort_values("Avg Error")


In [5]:
def query_claude_aws(prompt, model_id="anthropic.claude-3-5-sonnet-20240620-v1:0", max_tokens=500):
    bedrock = boto3.client("bedrock-runtime")

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": 0.5,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = bedrock.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    response_body = json.loads(response["body"].read())
    message_content = response_body["content"][0]["text"]

    try:
        # Try to parse the returned JSON text
        parsed_response = json.loads(message_content)
        return parsed_response
    except json.JSONDecodeError:
        # If parsing fails, return raw text in a fallback format
        return {
            "predicted_rate": "Unknown",
            "confidence": "Unknown",
            "explanation": message_content[:200] + "..."  # truncate long response
        }


In [6]:

def run_validation(fed_funds_df, economic_df, n):
    """
    Runs validation to compare Claude's predictions with actual Fed Funds Target Rate.

    Parameters:
        fed_funds_df (DataFrame): Historical daily Fed Funds Target Rate data.
        economic_df (DataFrame): Historical economic indicators.
        n (int): Number of months of past data to include.

    Returns:
        DataFrame: Results comparing predictions with actual values.
    """
    results = []

    for _, row in tqdm(fed_funds_df.iterrows(), total=len(fed_funds_df), desc="Validating Fed Rate Predictions"):
        target_date = row["Date"]
        actual_rate = row["Target_Rate"]  # The actual Fed funds rate

        # Find the latest available economic data before the target date
        past_data = economic_df[economic_df["Date"] < target_date].sort_values("Date", ascending=False)

        if past_data.empty:
            continue  # Skip if no past data is available

        # Select the last n months of economic data
        filtered_data = past_data.head(n)

        # Generate the prompt
        prompt = format_prompt(filtered_data, target_date, n)

        # Query Claude 3 on AWS Bedrock
        response = query_claude_aws(prompt)

        if isinstance(response, list) and len(response) > 0:
            response = response[0]  # Extract first item if it's a list

        # Store results
        results.append({
            "Date": target_date,
            "Actual_Fed_Funds_Rate": actual_rate,
            "Predicted_Fed_Funds_Rate": response.get("predicted_rate", "Unknown"),
            "Confidence": response.get("confidence", "Unknown"),
            "Explanation": response.get("explanation", "Unknown")
        })

    return pd.DataFrame(results)


In [ ]:
economic_df = load_economic_data("Data_cleaned.csv")
fed_funds_df = load_fed_funds_data("Monthly_Fed_Funds_Target.csv")

# 🚀 Run Prompt Evaluation
prompt_search_results = search_best_prompt_variants(prompt_variants, economic_df, fed_funds_df, n=6)

# Display top results
import pandas as pd
pd.set_option('display.max_colwidth', 180)
prompt_search_results.reset_index(drop=True).head(5)


In [ ]:
if __name__ == "__main__":
    # Load datasets
    economic_df = load_economic_data("Data_cleaned.csv")
    fed_funds_df = load_fed_funds_data("Monthly_Fed_Funds_Target.csv")

    n = 6  # Number of months of historical data for each prediction

    # Run validation
    results_df = run_validation(fed_funds_df, economic_df, n)

    # Save results
    results_df.to_csv("Claude_Fed_Funds_Validation.csv", index=False)



Validating Fed Rate Predictions:  52%|█████▏    | 58/112 [10:40<09:56, 11.04s/it]


KeyboardInterrupt: 

# 📈 Plot: Actual vs Claude-Predicted Fed Funds Target Rate
import matplotlib.pyplot as plt

# Make sure rates are numeric
results_df['Actual_Fed_Funds_Rate'] = pd.to_numeric(results_df['Actual_Fed_Funds_Rate'], errors='coerce')
results_df['Predicted_Fed_Funds_Rate'] = pd.to_numeric(results_df['Predicted_Fed_Funds_Rate'], errors='coerce')

# Drop rows with missing or invalid predictions
plot_df = results_df.dropna(subset=['Actual_Fed_Funds_Rate', 'Predicted_Fed_Funds_Rate'])

# Plot
plt.figure(figsize=(10, 5))
plt.plot(plot_df['Date'], plot_df['Actual_Fed_Funds_Rate'], label='Actual Rate', marker='o')
plt.plot(plot_df['Date'], plot_df['Predicted_Fed_Funds_Rate'], label='Claude Predicted Rate', marker='x')
plt.xlabel('Date')
plt.ylabel('Fed Funds Rate (%)')
plt.title('Actual vs Claude-Predicted Fed Funds Target Rate')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# # ⚙️ DEBUG: Run Claude prediction for just one test date

# # Pick a test date
# test_date = pd.Timestamp("2016-01-01")
# n_months = 6

# # Filter data
# past_data = economic_df[economic_df["Date"] < test_date].sort_values("Date", ascending=False).head(n_months)

# # Format prompt
# test_prompt = format_prompt(past_data, test_date, n_months)
# print("📝 Prompt sent to Claude:\n", test_prompt)

# # Call Claude
# test_response = query_claude_aws(test_prompt)
# print("\n📨 Raw response from Claude:\n", test_response)

# # If it's a list, extract first item
# if isinstance(test_response, list) and len(test_response) > 0:
#     test_response = test_response[0]

# # Show parsed parts
# print("\n✅ Parsed prediction:")
# print("Predicted Rate:", test_response.get("predicted_rate", "Unknown"))
# print("Confidence:", test_response.get("confidence", "Unknown"))
# print("Explanation:", test_response.get("explanation", "Unknown"))
